# Shortest Path

In [ ]:
from __future__ import annotations

import os

import h5py
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from tqdm import tqdm

from graph_prep import short_path, transforms

In [ ]:
main_dir = os.path.dirname(os.getcwd())
forward_link_path = os.path.join(main_dir, "data/output/enwiki_forward_links.hdf5")
matrix_info_path = os.path.join(main_dir, "data/output/enwiki_matrix_index_info.tsv")

In [ ]:
with h5py.File(forward_link_path, "r") as forw_f:
    forward_locs = forw_f["forward_locs"][:]
    forward_vals = forw_f["forward_vals"][:]

In [ ]:
backward_locs, backward_vals = transforms.swap_graph_direction(
    forward_locs, forward_vals, verbose=True
)

In [ ]:
matrix_vals_df = pd.read_csv(matrix_info_path, sep="\t")

### Get a couple example paths

In [ ]:
path_dest_list = [[1, 1125003], [12345, 543214], [12243, 78778], [10, 500]]

path_list = []

for path_dest in path_dest_list:
    path_list.append(
        short_path.path_one_way(
            path_dest[0],
            path_dest[1],
            forward_locs,
            forward_vals,
            backward_locs,
            backward_vals,
        )
    )

for path in path_list:
    for page in path[:-1]:
        print(matrix_vals_df.iloc[page - 1]["PageTitle"], end=" -> ")

    print(matrix_vals_df.iloc[path[-1] - 1]["PageTitle"])

### Path Histograms

In [ ]:
def plot_graph(steps_list: list[int], language: str, max_steps=10) -> None:
    plt.figure(dpi=200)
    plt.hist(
        steps_list,
        bins=np.arange(0, max_steps + 1, 1) - 0.5,
        rwidth=0.8,
        density=True,
        zorder=10,
        color="0.3",
    )

    plt.xlabel("Number of Steps")
    plt.ylabel("Fraction of Pairs of Pages")
    plt.title(f"Steps for Shortest Path {language} Wikipedia")

    plt.xticks(
        np.arange(0, max_steps, 1),
        labels=["NOT\nFOUND", *list(np.arange(1, max_steps, 1).astype(str))],
    )

    plt.grid()

    plt.show()

In [ ]:
def generate_steps_list(
    number_paths: int,
    max_steps: int,
    for_locs: np.ndarray,
    for_vals: np.ndarray,
    back_locs: np.ndarray,
    back_vals: np.ndarray,
    make_bad_zero: bool = True,
) -> np.ndarray:
    steps_list = []

    for sp, tp in tqdm(
        np.random.randint(1, for_locs.shape[0], (number_paths, 2)),
        desc="Generating Steps List",
    ):
        steps_list.append(
            short_path.steps_two_way(
                sp, tp, for_locs, for_vals, back_locs, back_vals, max_steps=max_steps
            )
        )

    steps_list = np.array(steps_list)

    if make_bad_zero:
        steps_list[steps_list == max_steps] = 0

    return steps_list

In [ ]:
en_steps_list = generate_steps_list(
    1000, 10, forward_locs, forward_vals, backward_locs, backward_vals
)

In [ ]:
plot_graph(en_steps_list, "English")